# Optimizers in Detail: SGD, Momentum, RMSProp, and Adam
Accompanies **Chapter 6: Optimizers in Detail**.

Standard gradient descent updates parameters along the negative gradient: $\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \nabla \mathcal{L}(\mathbf{w}_t)$.
On loss surfaces where curvature differs drastically across dimensions (anisotropic canyons), the gradient points almost orthogonally to the direction of the minimum, causing destructive oscillations and slow convergence.

This notebook simulates optimization paths on an anisotropic quadratic surface:
$$\mathcal{L}(w_1, w_2) = 0.1 w_1^2 + 2.0 w_2^2$$
The condition number of the Hessian $\mathbf{H} = \begin{bmatrix} 0.2 & 0 \\ 0 & 4.0 \end{bmatrix}$ is $\kappa = 4.0 / 0.2 = 20$, creating an ill-conditioned ravine.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def loss(w1, w2):
    return 0.1 * (w1 ** 2) + 2.0 * (w2 ** 2)

def gradient(w1, w2):
    return np.array([0.2 * w1, 4.0 * w2])

### Vanilla Stochastic Gradient Descent (SGD)
With a high learning rate, SGD oscillates violently along the steep $w_2$ axis while making minimal progress along the gentle $w_1$ valley floor.

In [ ]:
def run_sgd(w_init, lr=0.38, steps=30):
    w = np.array(w_init, dtype=float)
    path = [w.copy()]
    for _ in range(steps):
        g = gradient(w[0], w[1])
        w -= lr * g
        path.append(w.copy())
    return np.array(path)

path_sgd = run_sgd([4.0, 2.5], lr=0.38, steps=25)

### Momentum: Damping Transverse Oscillations
Momentum accumulates an exponentially decaying moving average of past gradients:
$$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \eta \nabla \mathcal{L}(\mathbf{w}_t), \quad \mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_t$$
Oscillating gradients along the steep axis cancel out, while consistent gradients along the valley accumulate velocity.

In [ ]:
def run_momentum(w_init, lr=0.12, beta=0.85, steps=30):
    w = np.array(w_init, dtype=float)
    v = np.zeros(2)
    path = [w.copy()]
    for _ in range(steps):
        g = gradient(w[0], w[1])
        v = beta * v + lr * g
        w -= v
        path.append(w.copy())
    return np.array(path)

path_mom = run_momentum([4.0, 2.5], lr=0.12, beta=0.85, steps=25)

### Adam: Adaptive Moment Estimation with Bias Correction
Adam tracks both the first moment (mean direction $\mathbf{m}_t$) and the uncentered second moment (variance $\mathbf{v}_t$), applying bias corrections to counter the initialization bias at $t=0$:
$$\hat{\mathbf{m}}_t = \frac{\mathbf{m}_t}{1 - \beta_1^t}, \quad \hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_2^t}$$
Parameters in steep directions receive smaller updates; parameters in flat directions receive larger updates.

In [ ]:
def run_adam(w_init, lr=0.45, beta1=0.9, beta2=0.999, eps=1e-8, steps=30):
    w = np.array(w_init, dtype=float)
    m = np.zeros(2)
    v = np.zeros(2)
    path = [w.copy()]
    for t in range(1, steps + 1):
        g = gradient(w[0], w[1])
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)
        m_hat = m / (1.0 - beta1 ** t)
        v_hat = v / (1.0 - beta2 ** t)
        w -= lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(w.copy())
    return np.array(path)

path_adam = run_adam([4.0, 2.5], lr=0.45, steps=25)

### Comparing Convergence Geometry on Loss Contours

In [ ]:
w1_grid = np.linspace(-5, 5, 200)
w2_grid = np.linspace(-3.5, 3.5, 200)
W1, W2 = np.meshgrid(w1_grid, w2_grid)
Z = loss(W1, W2)

plt.figure(figsize=(10, 5.5))
plt.contour(W1, W2, Z, levels=np.logspace(-1, 2, 25), cmap='viridis', alpha=0.5)
plt.plot(path_sgd[:, 0], path_sgd[:, 1], 'r.-', label='SGD: High oscillation across steep walls', linewidth=1.5)
plt.plot(path_mom[:, 0], path_mom[:, 1], 'b.-', label='Momentum: Damped transverse oscillation', linewidth=1.5)
plt.plot(path_adam[:, 0], path_adam[:, 1], 'g.-', label='Adam: Adaptive coordinate scaling', linewidth=2.0)
plt.scatter([0], [0], color='red', marker='*', s=150, zorder=5, label='Optimum (0, 0)')

plt.xlabel('$w_1$ (Flat Curvature)', fontsize=11)
plt.ylabel('$w_2$ (Steep Curvature)', fontsize=11)
plt.title('Optimization Trajectories on Anisotropic Quadratic Surface', fontsize=12, fontweight='bold')
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()